# Vietnamese to Classical Chinese machine translation: E1-E3 on Kaggle

This private, end-to-end notebook follows **CRISP-DM** from problem definition through deployment. It audits and visualizes the restricted DVSKTT corpus, demonstrates the exact preprocessing used by each model family, trains or restores E1/E2/custom-E3, evaluates all systems on the same gold splits, and exports reproducible private artifacts.

> **Privacy:** real corpus examples and predictions are displayed when `SHOW_PRIVATE_EXAMPLES=True`. Keep the notebook, outputs, datasets, checkpoints, and archives private.


## CRISP-DM roadmap

1. **Business understanding** — define the translation objective, experimental questions, constraints, and success criteria.
2. **Data understanding** — verify provenance, sizes, quality, leakage, lengths, vocabulary, duplicates, and knowledge coverage.
3. **Data preparation** — show the real normalized, model-specific representations and build Fairseq binaries.
4. **Modeling** — train/restore E1, resume E2 from epoch 3 to exactly epoch 5, and train/restore custom E3.
5. **Evaluation** — compare validation/test metrics, paired-bootstrap uncertainty, output lengths, examples, and manual errors.
6. **Deployment** — package manifests, predictions, metrics, figures, tables, checkpoints, and recovery archives.

The notebook deliberately runs E1, E2, and custom E3 as separate recoverable stages. Official E3 cannot be reconstructed from the supplied materials, so the knowledge-augmented system is always labelled **custom E3** rather than presented as the official system.


## 0. Kaggle setup and reproducibility controls

Use a Kaggle GPU runtime with Internet enabled and attach: (1) the complete private corpus and (2) the private E2 archive containing the epoch-3 Trainer checkpoint. Optional completed E1/E3 stage archives and an adjudicated `manual_sample.jsonl` can also be attached. The source repository is cloned at a recorded commit; all mutable work stays in `/kaggle/temp`, and only zip archives are written to `/kaggle/working`.


### Optional: acquire one input bundle from Google Drive

The preferred route is still a **private Kaggle Dataset** added through the notebook's Input pane. If that is impractical, upload one zip bundle to Google Drive, set the share mode needed for an unauthenticated download, paste its URL below, and provide its SHA-256 checksum. The cell downloads into disposable `/kaggle/temp` storage—not `/kaggle/working`—then the normal input scanner treats the extracted directory like any other input.

Expected bundle contents are the complete `zh-vi/` corpus plus restorable `checkpoint/`, `work/`, predictions, and metrics for E1, E2, and custom E3. An optional `manual_sample.jsonl` and `Error_Analysis.py` may also be included. Do not include the full Git repository; the notebook clones it separately. An anyone-with-link Drive URL weakens privacy, so use this fallback only if the data owner's rules permit it.


In [ ]:
from pathlib import Path
import hashlib
import shutil
import subprocess
import sys

# Leave both values empty when using a private Kaggle Dataset under /kaggle/input.
GOOGLE_DRIVE_BUNDLE_URL = "https://drive.google.com/file/d/1OKKFA88xnesQOodbi8ZXzYL1tpg2yxh6/view?usp=sharing"
GOOGLE_DRIVE_BUNDLE_SHA256 = "2446f8528eda7ac957742099585b880cf460d54aec2971af3899f364fa6ff818"

DRIVE_INPUT_ROOT = None
if GOOGLE_DRIVE_BUNDLE_URL.strip():
    if len(GOOGLE_DRIVE_BUNDLE_SHA256.strip()) != 64:
        raise ValueError("Set the bundle's 64-character SHA-256 checksum before downloading")

    drive_download_dir = Path("/kaggle/temp/group10-e1-e3/drive-download")
    drive_extract_dir = Path("/kaggle/temp/group10-e1-e3/drive-input")
    drive_download_dir.mkdir(parents=True, exist_ok=True)
    drive_extract_dir.mkdir(parents=True, exist_ok=True)
    bundle_path = drive_download_dir / "group10_kaggle_input_bundle.zip"

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "gdown==5.2.0"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "gdown", "--fuzzy", GOOGLE_DRIVE_BUNDLE_URL, "-O", str(bundle_path)],
        check=True,
    )

    digest = hashlib.sha256()
    with bundle_path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    actual_sha256 = digest.hexdigest()
    if actual_sha256.lower() != GOOGLE_DRIVE_BUNDLE_SHA256.strip().lower():
        raise RuntimeError({"expected_sha256": GOOGLE_DRIVE_BUNDLE_SHA256, "actual_sha256": actual_sha256})

    shutil.unpack_archive(str(bundle_path), str(drive_extract_dir))
    DRIVE_INPUT_ROOT = drive_extract_dir
    print("Verified and extracted Google Drive bundle to", DRIVE_INPUT_ROOT)
else:
    print("Google Drive fallback disabled; using attached Kaggle inputs.")


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys

# User-facing controls. Defaults execute the complete workflow.
REPO_URL = "https://github.com/ngnquanq/nlp_project.git"
REPO_REF = "feature/moses-substitute"
RUN_E1 = True
RUN_E2 = True
RUN_E3_CUSTOM = True
RESTORE_COMPLETED_STAGES = True
E2_TARGET_EPOCHS = 5
SHOW_PRIVATE_EXAMPLES = True
EDA_SAMPLE_SEED = 42
QUALITATIVE_SAMPLE_SIZE = 5
FIGURE_DPI = 140

KAGGLE_INPUT = Path("/kaggle/input")
KAGGLE_WORKING = Path("/kaggle/working")
KAGGLE_TEMP = Path("/kaggle/temp/group10-e1-e3")
REPO_DIR = KAGGLE_TEMP / "repo"
RESTORE_DIR = KAGGLE_TEMP / "restored"
STAGE_DIR = KAGGLE_TEMP / "stages"
FAIRSEQ_ENV = KAGGLE_TEMP / "fairseq-env"

E1_ID = "e1_fairseq_vi_zh_v1"
E2_ID = "e2_qwen3_8b_qlora_vi_zh_v1"
E3_ID = "e3_custom_fairseq_knowledge_vi_zh_v1"

for path in (KAGGLE_TEMP, RESTORE_DIR, STAGE_DIR):
    path.mkdir(parents=True, exist_ok=True)

os.environ.update({
    "CUDA_VISIBLE_DEVICES": "0",
    "HF_HOME": str(KAGGLE_TEMP / "hf-cache"),
    "HF_HUB_DISABLE_TELEMETRY": "1",
    "TOKENIZERS_PARALLELISM": "false",
    "PYTHONNOUSERSITE": "1",
})

def run_command(*args, env=None, cwd=None):
    command = [str(arg) for arg in args]
    print("+", " ".join(command), flush=True)
    merged = os.environ.copy()
    if env:
        merged.update({str(key): str(value) for key, value in env.items()})
    return subprocess.run(command, cwd=cwd or REPO_DIR, env=merged, check=True)

def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"Existing {REPO_DIR} is not a Git clone; use a fresh Kaggle session")
    existing_origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=REPO_DIR, text=True
    ).strip()
    if existing_origin.rstrip("/").removesuffix(".git") != REPO_URL.rstrip("/").removesuffix(".git"):
        raise RuntimeError({"expected_origin": REPO_URL, "existing_origin": existing_origin})
    print("Reusing existing verified repository clone:", REPO_DIR)
else:
    run_command("git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, REPO_DIR, cwd=KAGGLE_TEMP)

RESOLVED_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True
).strip()
os.chdir(REPO_DIR)
CODE_DIR = REPO_DIR / "code"
os.environ["PYTHONPATH"] = str(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
print("Registered project imports from:", CODE_DIR)
print("Resolved source commit:", RESOLVED_COMMIT)

ANALYSIS_DIR = REPO_DIR / "analysis"
FIGURE_DIR = ANALYSIS_DIR / "figures"
TABLE_DIR = ANALYSIS_DIR / "tables"
for path in (FIGURE_DIR, TABLE_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Inputs may come from Kaggle's Input pane or the optional verified Drive bundle.
INPUT_ROOTS = [KAGGLE_INPUT]
if DRIVE_INPUT_ROOT is not None:
    INPUT_ROOTS.append(DRIVE_INPUT_ROOT)

# Stage archives from an earlier run are unpacked into disposable recovery storage.
for input_root in INPUT_ROOTS:
    for archive in sorted(input_root.rglob("group10_*_results.zip")):
        destination = RESTORE_DIR / archive.stem
        destination.mkdir(parents=True, exist_ok=True)
        shutil.unpack_archive(str(archive), str(destination))

SEARCH_ROOTS = tuple(INPUT_ROOTS) + (RESTORE_DIR,)

def experiment_manifests(experiment_id):
    found = []
    for root in SEARCH_ROOTS:
        found.extend(
            path for path in root.rglob("run_manifest.json")
            if path.parent.name == experiment_id
        )
    unique = {path.resolve(): path for path in found}
    return sorted(unique.values())

def manifest_repo_root(manifest_path):
    # <root>/work/<experiment_id>/run_manifest.json
    root = manifest_path.parents[2]
    if not (root / "work" / manifest_path.parent.name / "run_manifest.json").is_file():
        raise RuntimeError(f"Unexpected artifact layout around {manifest_path}")
    return root

EXPECTED_DATA_FILES = (
    "train/DVSKTT_self_aligned_cn_sv_vi_20240927_cleaned_official.no_punct.line.train.vi",
    "train/DVSKTT_self_aligned_cn_sv_vi_20240927_cleaned_official.no_punct.line.train.cn",
    "val/DVSKTT_manually_aligned_sent_cleaned_gold_eval.20240829.no_punct.val.vi",
    "val/DVSKTT_manually_aligned_sent_cleaned_gold_eval.20240829.no_punct.val.cn",
    "test/DVSKTT_manually_aligned_sent_cleaned_gold_eval.20240829.no_punct.test.vi",
    "test/DVSKTT_manually_aligned_sent_cleaned_gold_eval.20240829.no_punct.test.cn",
    "knowledge.json",
)
data_roots = []
for root in SEARCH_ROOTS:
    for knowledge in root.rglob("knowledge.json"):
        candidate = knowledge.parent
        if all((candidate / relative).is_file() for relative in EXPECTED_DATA_FILES):
            data_roots.append(candidate)
if not data_roots:
    raise RuntimeError("No complete private zh-vi corpus was found under /kaggle/input")

def data_signature(root):
    return tuple(sha256_file(root / relative) for relative in EXPECTED_DATA_FILES)

signatures = {data_signature(root) for root in data_roots}
if len(signatures) != 1:
    raise RuntimeError(f"Attached inputs contain conflicting corpus copies: {data_roots}")
DATA_SOURCE = sorted(data_roots, key=lambda path: (len(path.parts), str(path)))[0]
DATA_DESTINATION = REPO_DIR / "zh-vi"
if DATA_DESTINATION.exists():
    if not all((DATA_DESTINATION / relative).is_file() for relative in EXPECTED_DATA_FILES):
        raise RuntimeError(f"Existing corpus copy is incomplete: {DATA_DESTINATION}")
    if data_signature(DATA_DESTINATION) != data_signature(DATA_SOURCE):
        raise RuntimeError("Existing corpus copy conflicts with the attached private input")
    print("Reusing existing fingerprint-verified corpus copy:", DATA_DESTINATION)
else:
    shutil.copytree(DATA_SOURCE, DATA_DESTINATION)
print(f"Validated {len(data_roots)} identical corpus location(s); using {DATA_SOURCE}")


In [ ]:
from importlib.metadata import version

E2_MANIFESTS = experiment_manifests(E2_ID)
if len(E2_MANIFESTS) != 1:
    raise RuntimeError(f"Expected exactly one attached E2 run manifest, found {E2_MANIFESTS}")
E2_SOURCE_MANIFEST = E2_MANIFESTS[0]
E2_SOURCE_ROOT = manifest_repo_root(E2_SOURCE_MANIFEST)
prior_manifest = read_json(E2_SOURCE_MANIFEST)
prior_config = dict(prior_manifest["config"])
prior_config.pop("_config_path", None)

expected_versions = {}
for line in (REPO_DIR / "requirements-llm.lock.txt").read_text(encoding="utf-8").splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        name, expected = line.split("==", 1)
        expected_versions[name] = expected
for name, saved in prior_manifest.get("packages", {}).items():
    if saved:
        expected_versions[name] = saved
install_args = [
    f"{name}=={saved}" for name, saved in sorted(expected_versions.items())
    if name.lower() != "torch"
]
run_command(sys.executable, "-m", "pip", "install", "--no-cache-dir", *install_args)
import yaml

# Kaggle manages the CUDA-enabled PyTorch build as part of its accelerator image.
# Pin every project package except torch; replacing torch here can silently remove CUDA.
managed_expected_versions = {
    name: saved for name, saved in expected_versions.items() if name.lower() != "torch"
}
actual_versions = {name: version(name) for name in managed_expected_versions}
if actual_versions != managed_expected_versions:
    raise RuntimeError({
        "expected_packages": managed_expected_versions,
        "actual_packages": actual_versions,
    })
print("Verified pinned project packages; Kaggle runtime manages torch:", version("torch"))

# The root-level automatic error-analysis script reads and writes Excel workbooks.
run_command(sys.executable, "-m", "pip", "install", "--no-cache-dir", "openpyxl==3.1.5")
if version("openpyxl") != "3.1.5":
    raise RuntimeError(f"Unexpected openpyxl version: {version('openpyxl')}")
ERROR_ANALYSIS_SCRIPT = REPO_DIR / "Error_Analysis.py"
if not ERROR_ANALYSIS_SCRIPT.is_file():
    supplied_scripts = sorted(
        path for root in SEARCH_ROOTS for path in root.rglob("Error_Analysis.py")
    )
    if not supplied_scripts:
        raise RuntimeError(
            "Error_Analysis.py is absent from both the cloned commit and attached input bundle."
        )
    script_hashes = {sha256_file(candidate) for candidate in supplied_scripts}
    if len(script_hashes) != 1:
        raise RuntimeError(f"Attached inputs contain conflicting Error_Analysis.py files: {supplied_scripts}")
    shutil.copy2(supplied_scripts[0], ERROR_ANALYSIS_SCRIPT)
    print("Recovered Error_Analysis.py from verified private input bundle")

import torch
if not torch.cuda.is_available():
    raise RuntimeError(
        f"CUDA is unavailable (torch={torch.__version__}). In Kaggle, open Settings, "
        "set Accelerator to GPU, start a fresh session, and run the notebook again."
    )
properties = torch.cuda.get_device_properties(0)
if properties.total_memory < 14 * 1024**3:
    raise RuntimeError(f"At least 14 GiB VRAM is required; found {properties.total_memory / 1024**3:.1f} GiB")
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0), f"{properties.total_memory / 1024**3:.1f} GiB")

checked_in = yaml.safe_load((REPO_DIR / "configs/e2_qwen3_qlora.yaml").read_text(encoding="utf-8"))
comparable_prior = json.loads(json.dumps(prior_config))
comparable_prior.pop("resume", None)
comparable_prior["training"]["epochs"] = checked_in["training"]["epochs"]
if comparable_prior != checked_in:
    raise RuntimeError("The attached E2 run does not match the checked-in E2 experiment contract")
if prior_manifest.get("resolved_model_revision") != checked_in["model"]["revision"]:
    raise RuntimeError("The attached E2 run resolved a different Qwen model revision")

run_command(sys.executable, "-m", "pytest", "-q", "-m", "not private_data", "tests")
run_command(sys.executable, "-m", "mt_pipeline", "data-audit", "--config", "configs/dataset.yaml", "--output", "metrics/data_audit.json")
from mt_pipeline.data import dataset_fingerprint
current_fingerprint = dataset_fingerprint("configs/dataset.yaml")
if prior_manifest.get("dataset") != current_fingerprint:
    raise RuntimeError("The attached E2 checkpoint was trained against a different corpus fingerprint")

prior_work = E2_SOURCE_ROOT / "work" / E2_ID
prior_adapter = E2_SOURCE_ROOT / "checkpoint" / E2_ID / "adapter"
trainer_checkpoints = []
for path in (prior_work / "trainer").glob("checkpoint-*"):
    try:
        trainer_checkpoints.append((int(path.name.rsplit("-", 1)[1]), path))
    except ValueError:
        pass
if not trainer_checkpoints:
    raise RuntimeError("The attached E2 output has no resumable Trainer checkpoint")
resume_source = max(trainer_checkpoints)[1]
resume_state = read_json(resume_source / "trainer_state.json")
source_epoch = float(resume_state["epoch"])
source_final_state = read_json(prior_work / "trainer" / "trainer_state.json")
completed_e2 = float(source_final_state["epoch"]) >= E2_TARGET_EPOCHS

required_resume = [
    resume_source / name
    for name in ("adapter_config.json", "trainer_state.json", "optimizer.pt", "scheduler.pt", "rng_state.pth")
]
weights = (resume_source / "adapter_model.safetensors", resume_source / "adapter_model.bin")
missing = [str(path) for path in required_resume if not path.exists()]
if not any(path.exists() for path in weights):
    missing.append("adapter_model.safetensors or adapter_model.bin")
if missing:
    raise RuntimeError({"missing_E2_resume_files": missing})
if not completed_e2 and abs(source_epoch - 3.0) > 1e-6:
    raise RuntimeError(f"Expected the latest resumable E2 checkpoint at epoch 3, found {source_epoch:g}")

destination_work = REPO_DIR / "work" / E2_ID
shutil.copytree(prior_work, destination_work, dirs_exist_ok=True)
for state_path in (destination_work / "trainer").rglob("trainer_state.json"):
    state = read_json(state_path)
    if state.get("best_model_checkpoint"):
        relocated_best = destination_work / "trainer" / Path(state["best_model_checkpoint"]).name
        if not relocated_best.is_dir():
            raise RuntimeError(f"Trainer state references a missing best checkpoint: {relocated_best}")
        state["best_model_checkpoint"] = str(relocated_best)
        state_path.write_text(json.dumps(state, indent=2, sort_keys=True) + "\n", encoding="utf-8")
if prior_adapter.is_dir():
    shutil.copytree(prior_adapter, REPO_DIR / "checkpoint" / E2_ID / "adapter", dirs_exist_ok=True)

if completed_e2:
    for split in ("val", "test"):
        for folder, suffix in (("predictions", "jsonl"), ("metrics", "json")):
            source_path = E2_SOURCE_ROOT / folder / f"{E2_ID}.{split}.{suffix}"
            if not source_path.is_file():
                raise RuntimeError(f"Completed attached E2 stage is missing {source_path}")
            destination_path = REPO_DIR / folder / source_path.name
            destination_path.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source_path, destination_path)

runtime_config = dict(prior_config)
runtime_config["training"] = dict(runtime_config["training"])
runtime_config["training"]["epochs"] = E2_TARGET_EPOCHS
runtime_config["resume"] = True
E2_CONFIG = REPO_DIR / "configs/e2_qwen3_qlora_kaggle_5epochs.yaml"
E2_CONFIG.write_text(yaml.safe_dump(runtime_config, allow_unicode=True, sort_keys=True), encoding="utf-8")

if completed_e2:
    print("A completed five-epoch E2 stage was attached; it will be verified instead of retrained.")
elif not RUN_E2:
    raise RuntimeError("RUN_E2=False but the attached E2 stage has not reached five epochs")
else:
    stale_selection = destination_work / "selection_frozen.json"
    if stale_selection.exists():
        stale_selection.unlink()
    print(f"Recovered {resume_source.name} at epoch {source_epoch:g}; fixed target is epoch {E2_TARGET_EPOCHS}.")


## 1. Business understanding

**Objective.** Translate lower-cased, tokenized Vietnamese historical text into punctuation-free Classical Chinese while preserving meaning, named entities, numbers, and temporal information.

**Stakeholders and use.** This is a Group 10 research comparison, not a production translation service. Researchers need a reproducible comparison; readers need transparent limitations; data owners require that the restricted corpus and all derived outputs remain private.

**Experimental questions.** E1 asks what a conventional Transformer learns from the parallel corpus. E2 asks whether Qwen3-8B adapted with QLoRA improves translation quality. Custom E3 asks whether deterministic dictionary retrieval helps the E1 architecture.

**Success criteria.** Every run must use the same immutable split fingerprints and stable sample IDs; validation alone selects checkpoints; the test split is evaluated once after selection; all 510 validation and 510 test rows must pass the prediction schema; BLEU and chrF++ must reproduce; pairwise differences must include a seeded bootstrap interval; and all artifacts must carry configuration, commit, and checksum provenance. Better metrics are evidence of performance on this dataset—not proof of general historical-language competence.


## 2. Data understanding

The DVSKTT dataset contains a self-aligned silver training split and manually aligned gold validation/test splits. Vietnamese is whitespace-tokenized; Classical Chinese targets are supplied character-separated. The audit above verifies hashes, aligned row counts, duplicates, replacement characters, cross-split overlap, and knowledge coverage before any model is trained.


In [ ]:
import random
from collections import Counter
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from mt_pipeline.data import load_dataset_config, read_split

plt.style.use("seaborn-v0_8-whitegrid")

def save_figure(filename):
    path = FIGURE_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=FIGURE_DPI, bbox_inches="tight")
    plt.show()
    print("Saved", path.relative_to(REPO_DIR))

dataset_config = load_dataset_config("configs/dataset.yaml")
audit = read_json(REPO_DIR / "metrics/data_audit.json")
split_rows = {split: read_split(dataset_config, split) for split in ("train", "val", "test")}

records = []
for split, rows in split_rows.items():
    for row in rows:
        records.append({
            "split": split,
            "sample_id": row.sample_id,
            "source": row.source,
            "reference": row.reference,
            "reference_tokenized": row.reference_tokenized,
            "source_tokens": len(row.source.split()),
            "target_characters": len(row.reference_tokenized.split()),
        })
corpus_df = pd.DataFrame(records)

split_summary = pd.DataFrame([
    {
        "split": split,
        "role": "optimization" if split == "train" else ("selection" if split == "val" else "final evaluation"),
        "quality": audit["splits"][split]["quality"],
        "pairs": details["parallel_pairs"],
        "duplicate_pairs": details["duplicate_pairs"],
        "replacement_rows": len(details["replacement_characters"]),
    }
    for split, details in audit["splits"].items()
])
split_summary.to_csv(TABLE_DIR / "dataset_split_summary.csv", index=False)
display(split_summary)

if SHOW_PRIVATE_EXAMPLES:
    examples = []
    for offset, split in enumerate(("train", "val", "test")):
        row = split_rows[split][random.Random(EDA_SAMPLE_SEED + offset).randrange(len(split_rows[split]))]
        examples.append({"split": split, "sample_id": row.sample_id, "Vietnamese source": row.source, "Classical Chinese reference": row.reference})
    display(pd.DataFrame(examples))
else:
    print("Private examples hidden. Set SHOW_PRIVATE_EXAMPLES=True only in a private notebook.")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
split_summary.plot.bar(x="split", y="pairs", legend=False, ax=axes[0], color=["#355070", "#6d597a", "#b56576"])
axes[0].set(title="Parallel pairs by split", ylabel="Sentence pairs", xlabel="")
for container in axes[0].containers:
    axes[0].bar_label(container, fmt="%d")

for split, color in zip(("train", "val", "test"), ("#355070", "#6d597a", "#b56576")):
    values = corpus_df.loc[corpus_df.split == split, "source_tokens"]
    axes[1].hist(values, bins=35, alpha=0.48, density=True, label=split, color=color)
axes[1].set(title="Vietnamese source-length distributions", xlabel="Whitespace tokens", ylabel="Density")
axes[1].legend()
save_figure("dataset_sizes_and_source_lengths.png")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for split, color in zip(("train", "val", "test"), ("#355070", "#6d597a", "#b56576")):
    values = corpus_df.loc[corpus_df.split == split, "target_characters"]
    axes[0].hist(values, bins=35, alpha=0.48, density=True, label=split, color=color)
axes[0].set(title="Classical Chinese target lengths", xlabel="Characters", ylabel="Density")
axes[0].legend()
hb = axes[1].hexbin(corpus_df.source_tokens, corpus_df.target_characters, gridsize=40, mincnt=1, cmap="viridis")
axes[1].set(title="Source-target length relationship", xlabel="Vietnamese tokens", ylabel="Chinese characters")
fig.colorbar(hb, ax=axes[1], label="Sentence-pair count")
save_figure("length_distributions_and_relationship.png")


In [ ]:
lexical_rows = []
for split in ("train", "val", "test"):
    details = audit["splits"][split]
    for side in ("source", "target"):
        lexical_rows.append({
            "split": split, "side": side,
            "tokens": details[side]["tokens"],
            "types": details[side]["types"],
            "hapax_types": details[side]["hapax_types"],
            "hapax_share": details[side]["hapax_types"] / max(details[side]["types"], 1),
        })
lexical_df = pd.DataFrame(lexical_rows)
lexical_df.to_csv(TABLE_DIR / "lexical_statistics.csv", index=False)

quality_rows = []
for split, details in audit["splits"].items():
    quality_rows.append({
        "split": split,
        "duplicate_pairs": details["duplicate_pairs"],
        "duplicate_sources": details["duplicate_sources"],
        "duplicate_targets": details["duplicate_targets"],
        "multi_target_sources": details["sources_with_multiple_targets"],
        "replacement_rows": len(details["replacement_characters"]),
    })
quality_df = pd.DataFrame(quality_rows)
quality_df.to_csv(TABLE_DIR / "data_quality_summary.csv", index=False)

overlap_df = pd.DataFrame([{"split_pair": key, **value} for key, value in audit["overlap"].items()])
overlap_df.to_csv(TABLE_DIR / "cross_split_overlap.csv", index=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
lexical_df.pivot(index="split", columns="side", values="types").plot.bar(ax=axes[0], color=["#457b9d", "#e76f51"])
axes[0].set(title="Vocabulary types", xlabel="", ylabel="Unique whitespace tokens")
lexical_df.pivot(index="split", columns="side", values="hapax_share").plot.bar(ax=axes[1], color=["#457b9d", "#e76f51"])
axes[1].set(title="Hapax share", xlabel="", ylabel="Share of types")
axes[1].yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
knowledge = audit["knowledge"]
uncovered = knowledge["corpus_target_characters"] - knowledge["covered_target_characters"]
axes[2].bar(["covered", "uncovered"], [knowledge["covered_target_characters"], uncovered], color=["#2a9d8f", "#e9c46a"])
axes[2].set(title=f"Knowledge coverage ({knowledge['coverage']:.1%})", ylabel="Distinct target characters")
save_figure("lexical_and_knowledge_coverage.png")

display(quality_df)
display(overlap_df)
print(f"Audit found {len(audit['issues'])} replacement-character rows and zero exact overlap when every overlap count above is zero.")


## 3. Data preparation

Raw files are treated as immutable. `read_split` normalizes whitespace and assigns deterministic row-order IDs. Fairseq receives the supplied Vietnamese tokens and character-spaced Chinese targets; Qwen receives a chat prompt and natural, unspaced Chinese target, while predictions are normalized back to character-spaced form for scoring. Custom E3 uses deterministic lookup, ranking candidates by match count, training-target frequency, then Unicode order, and appends only candidates that fit the encoder budget.


In [ ]:
import yaml
from mt_pipeline.knowledge import KnowledgeAugmenter
from mt_pipeline.normalize import normalize_whitespace, detokenize_chinese, score_form_chinese

e2_checked_config = yaml.safe_load((REPO_DIR / "configs/e2_qwen3_qlora.yaml").read_text(encoding="utf-8"))
e3_checked_config = yaml.safe_load((REPO_DIR / "configs/e3_custom_fairseq_knowledge.yaml").read_text(encoding="utf-8"))
demo = split_rows["train"][EDA_SAMPLE_SEED % len(split_rows["train"])]
augmented = KnowledgeAugmenter(e3_checked_config).augment(demo.source)
qwen_prompt = e2_checked_config["prompt"]["system"] + "\n\n" + e2_checked_config["prompt"]["user_template"].format(source=demo.source)
prep_demo = pd.DataFrame([
    {"representation": "Stable ID", "value": demo.sample_id},
    {"representation": "Normalized Vietnamese", "value": normalize_whitespace(demo.source)},
    {"representation": "Fairseq target", "value": demo.reference_tokenized},
    {"representation": "Qwen target", "value": detokenize_chinese(demo.reference_tokenized)},
    {"representation": "Shared scoring form", "value": score_form_chinese(demo.reference)},
    {"representation": "Qwen chat prompt", "value": qwen_prompt},
    {"representation": "Custom E3 source", "value": augmented.source},
])
if SHOW_PRIVATE_EXAMPLES:
    display(prep_demo)
else:
    print("Transformation example hidden because SHOW_PRIVATE_EXAMPLES=False.")
print({"E3_candidate_count": augmented.candidate_count, "E3_matched_spans": augmented.matched_spans})


In [ ]:
from mt_pipeline.io_utils import sha256_tree

def attached_fairseq_stage_ready(experiment_id, config_name):
    manifests = experiment_manifests(experiment_id)
    if len(manifests) != 1:
        return False
    source_root = manifest_repo_root(manifests[0])
    checkpoint = source_root / "checkpoint" / experiment_id / "checkpoint_best.pt"
    selection_path = source_root / "work" / experiment_id / "selection_frozen.json"
    required = [
        checkpoint, selection_path,
        source_root / "predictions" / f"{experiment_id}.val.jsonl",
        source_root / "predictions" / f"{experiment_id}.test.jsonl",
        source_root / "metrics" / f"{experiment_id}.val.json",
        source_root / "metrics" / f"{experiment_id}.test.json",
    ]
    if not all(candidate.is_file() for candidate in required):
        return False
    selection = read_json(selection_path)
    config_path = REPO_DIR / "configs" / config_name
    if selection.get("config_sha256") != sha256_file(config_path):
        return False
    if selection.get("checkpoint_sha256") != sha256_file(checkpoint):
        return False
    for split in ("val", "test"):
        prediction = source_root / "predictions" / f"{experiment_id}.{split}.jsonl"
        if sum(bool(line) for line in prediction.read_text(encoding="utf-8").splitlines()) != 510:
            return False
    return True

E1_RESTORE_READY = attached_fairseq_stage_ready(E1_ID, "e1_fairseq.yaml")
E3_RESTORE_READY = attached_fairseq_stage_ready(E3_ID, "e3_custom_fairseq_knowledge.yaml")
NEED_FAIRSEQ_RUNTIME = not (E1_RESTORE_READY and E3_RESTORE_READY)
print({
    "E1_restore_ready": E1_RESTORE_READY,
    "custom_E3_restore_ready": E3_RESTORE_READY,
    "create_Fairseq_runtime": NEED_FAIRSEQ_RUNTIME,
})

FAIRSEQ_PYTHON = None
FAIRSEQ_ENV_VARS = {}
if NEED_FAIRSEQ_RUNTIME:
    import urllib.request

    # Kaggle images no longer guarantee a `conda` executable. Bootstrap a pinned,
    # standalone micromamba binary and use it to solve the existing Python 3.10 spec.
    MICROMAMBA_VERSION = "2.8.1-0"
    MICROMAMBA_URL = (
        "https://github.com/mamba-org/micromamba-releases/releases/download/"
        f"{MICROMAMBA_VERSION}/micromamba-linux-64"
    )
    MICROMAMBA_SHA256 = "9689782d863c05a1bf5d2d371ba527104e7a4eb4310c1637d8653b751aed9c82"
    MICROMAMBA_BIN = KAGGLE_TEMP / "tools/micromamba"
    MAMBA_ROOT_PREFIX = KAGGLE_TEMP / "micromamba-root"

    if not MICROMAMBA_BIN.is_file():
        MICROMAMBA_BIN.parent.mkdir(parents=True, exist_ok=True)
        temporary_binary = MICROMAMBA_BIN.with_suffix(".download")
        with urllib.request.urlopen(MICROMAMBA_URL, timeout=120) as response, temporary_binary.open("wb") as handle:
            shutil.copyfileobj(response, handle)
        if sha256_file(temporary_binary) != MICROMAMBA_SHA256:
            temporary_binary.unlink(missing_ok=True)
            raise RuntimeError("Downloaded micromamba binary failed SHA-256 verification")
        temporary_binary.chmod(0o755)
        temporary_binary.replace(MICROMAMBA_BIN)
    elif sha256_file(MICROMAMBA_BIN) != MICROMAMBA_SHA256:
        raise RuntimeError(f"Existing micromamba binary has an unexpected checksum: {MICROMAMBA_BIN}")

    FAIRSEQ_PYTHON = FAIRSEQ_ENV / "bin/python"
    if FAIRSEQ_PYTHON.is_file():
        print("Reusing existing Fairseq environment:", FAIRSEQ_ENV)
    else:
        run_command(
            MICROMAMBA_BIN,
            "create", "--yes",
            "--prefix", FAIRSEQ_ENV,
            "--file", REPO_DIR / "environments/fairseq.yml",
            env={"MAMBA_ROOT_PREFIX": MAMBA_ROOT_PREFIX},
            cwd=KAGGLE_TEMP,
        )
    if not FAIRSEQ_PYTHON.is_file():
        raise RuntimeError(f"Micromamba did not create the expected Python executable: {FAIRSEQ_PYTHON}")

    FAIRSEQ_ENV_VARS = {
        "PYTHONPATH": REPO_DIR / "code",
        "PYTHONNOUSERSITE": "1",
        "CUDA_VISIBLE_DEVICES": "0",
        "CONDA_PREFIX": FAIRSEQ_ENV,
        "CONDA_DEFAULT_ENV": FAIRSEQ_ENV.name,
        "PATH": str(FAIRSEQ_ENV / "bin") + os.pathsep + os.environ["PATH"],
    }
    run_command(FAIRSEQ_PYTHON, "-c", "import fairseq, torch; assert torch.cuda.is_available(); print(fairseq.__version__, torch.__version__, torch.cuda.get_device_name(0))", env=FAIRSEQ_ENV_VARS)

def restore_completed_stage(experiment_id, config_path):
    if not RESTORE_COMPLETED_STAGES:
        return False
    manifests = experiment_manifests(experiment_id)
    if not manifests:
        return False
    if len(manifests) != 1:
        raise RuntimeError(f"Expected at most one attached {experiment_id} manifest, found {manifests}")
    source_root = manifest_repo_root(manifests[0])
    required = [
        source_root / "checkpoint" / experiment_id / "checkpoint_best.pt",
        source_root / "work" / experiment_id / "selection_frozen.json",
        source_root / "predictions" / f"{experiment_id}.val.jsonl",
        source_root / "predictions" / f"{experiment_id}.test.jsonl",
        source_root / "metrics" / f"{experiment_id}.val.json",
        source_root / "metrics" / f"{experiment_id}.test.json",
    ]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise RuntimeError({"incomplete_attached_fairseq_stage": experiment_id, "missing": missing})
    for relative in (f"checkpoint/{experiment_id}", f"work/{experiment_id}"):
        shutil.copytree(source_root / relative, REPO_DIR / relative, dirs_exist_ok=True)
    for split in ("val", "test"):
        for folder, suffix in (("predictions", "jsonl"), ("metrics", "json")):
            source = source_root / folder / f"{experiment_id}.{split}.{suffix}"
            destination = REPO_DIR / folder / source.name
            destination.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source, destination)
    validate_complete_stage(config_path, experiment_id)
    print("Restored and verified", experiment_id)
    return True

def run_fairseq_stage(label, experiment_id, config_name, enabled, backend):
    config_path = REPO_DIR / "configs" / config_name
    local_required = [
        REPO_DIR / "work" / experiment_id / "selection_frozen.json",
        REPO_DIR / "checkpoint" / experiment_id / "checkpoint_best.pt",
        prediction_path(experiment_id, "val"), prediction_path(experiment_id, "test"),
        metric_path(experiment_id, "val"), metric_path(experiment_id, "test"),
    ]
    if all(candidate.exists() for candidate in local_required):
        try:
            validate_complete_stage(config_path, experiment_id)
        except (RuntimeError, ValueError, FileNotFoundError) as error:
            print(f"Existing local {experiment_id} is not reusable: {error}")
        else:
            print("Reusing verified stage from this Kaggle session:", experiment_id)
            return stage_archive(label, experiment_id, config_path, backend)

    restored = restore_completed_stage(experiment_id, config_path)
    if not restored:
        if not enabled:
            raise RuntimeError(f"{experiment_id} is incomplete while its run toggle is disabled")
        try:
            run_command(FAIRSEQ_PYTHON, "-m", "mt_pipeline", "train", "--config", config_path, env=FAIRSEQ_ENV_VARS)
        except subprocess.CalledProcessError:
            partial_archive(label, experiment_id, config_path, backend)
            raise
        run_evaluation_sequence(config_path, experiment_id, FAIRSEQ_PYTHON, FAIRSEQ_ENV_VARS)
        validate_complete_stage(config_path, experiment_id)
    return stage_archive(label, experiment_id, config_path, backend)


In [ ]:
# Build Fairseq inputs only when a Fairseq stage is missing. Complete attached
# stages already contain their immutable data-bin dictionaries and preprocessing report.
E1_CONFIG = REPO_DIR / "configs/e1_fairseq.yaml"
E3_CONFIG = REPO_DIR / "configs/e3_custom_fairseq_knowledge.yaml"
if NEED_FAIRSEQ_RUNTIME:
    for config_path in (E1_CONFIG, E3_CONFIG):
        run_command(FAIRSEQ_PYTHON, "-m", "mt_pipeline", "prepare-fairseq", "--config", config_path, env=FAIRSEQ_ENV_VARS)
else:
    print("Skipping Fairseq environment creation and preprocessing: E1 and custom E3 are restorable.")

prep_rows = []
for label, experiment_id in (("E1", E1_ID), ("Custom E3", E3_ID)):
    local_work = REPO_DIR / "work" / experiment_id
    if (local_work / "data-bin").is_dir():
        work = local_work
    else:
        manifests = experiment_manifests(experiment_id)
        if len(manifests) != 1:
            raise RuntimeError(f"Expected one preprocessing artifact source for {experiment_id}")
        work = manifest_repo_root(manifests[0]) / "work" / experiment_id
    dictionaries = sorted((work / "data-bin").glob("dict.*.txt"))
    row = {"experiment": label, "artifact_source": "attached" if work != local_work else "prepared"}
    for dictionary in dictionaries:
        row[dictionary.stem] = sum(1 for line in dictionary.read_text(encoding="utf-8").splitlines() if line)
    report = work / "knowledge_augmentation.json"
    if report.exists():
        payload = read_json(report)
        row.update({f"augmentation_{key}": value for key, value in payload.items() if isinstance(value, (int, float, str, bool))})
    prep_rows.append(row)
preprocessing_df = pd.DataFrame(prep_rows)
preprocessing_df.to_csv(TABLE_DIR / "fairseq_preprocessing_summary.csv", index=False)
display(preprocessing_df)


## 4. Modeling

E1 and custom E3 use the same 6-layer Fairseq Transformer and optimization contract; their controlled difference is knowledge augmentation. E2 uses revision-pinned Qwen3-8B with 4-bit NF4 quantization and rank-16 QLoRA. The attached E2 epoch-3 state includes optimizer, scheduler, RNG, and adapter state and is resumed to **exactly five epochs**. Validation selects checkpoints; test references never influence selection.


In [ ]:
def prediction_path(experiment_id, split):
    return REPO_DIR / "predictions" / f"{experiment_id}.{split}.jsonl"

def metric_path(experiment_id, split):
    return REPO_DIR / "metrics" / f"{experiment_id}.{split}.json"

def run_evaluation_sequence(config_path, experiment_id, model_python, model_env=None):
    run_command(model_python, "-m", "mt_pipeline", "predict", "--config", config_path, "--split", "val", env=model_env)
    run_command(sys.executable, "-m", "mt_pipeline", "evaluate", "--predictions", prediction_path(experiment_id, "val"), "--output", metric_path(experiment_id, "val"))
    run_command(sys.executable, "-m", "mt_pipeline", "freeze-selection", "--config", config_path, "--validation-predictions", prediction_path(experiment_id, "val"), "--validation-metrics", metric_path(experiment_id, "val"))
    run_command(model_python, "-m", "mt_pipeline", "predict", "--config", config_path, "--split", "test", env=model_env)
    run_command(sys.executable, "-m", "mt_pipeline", "evaluate", "--predictions", prediction_path(experiment_id, "test"), "--output", metric_path(experiment_id, "test"))

def validate_complete_stage(config_path, experiment_id):
    from mt_pipeline.evaluation import metrics_equivalent, protocol_from_metrics
    from mt_pipeline.freeze import ensure_selection_frozen
    ensure_selection_frozen(config_path)
    for split in ("val", "test"):
        rows = [line for line in prediction_path(experiment_id, split).read_text(encoding="utf-8").splitlines() if line]
        if len(rows) != 510:
            raise RuntimeError(f"{experiment_id} {split} has {len(rows)} rows, expected 510")
        saved = read_json(metric_path(experiment_id, split))
        scratch = KAGGLE_TEMP / f"{experiment_id}.{split}.recomputed.json"
        run_command(sys.executable, "-m", "mt_pipeline", "evaluate", "--predictions", prediction_path(experiment_id, split), "--protocol", protocol_from_metrics(saved["metrics"]), "--output", scratch)
        recomputed = read_json(scratch).get("metrics")
        stored = saved.get("metrics")
        if not isinstance(stored, dict) or not isinstance(recomputed, dict) or not metrics_equivalent(stored, recomputed):
            raise RuntimeError({
                "stored_metrics_do_not_reproduce": f"{experiment_id} {split}",
                "stored": stored,
                "recomputed": recomputed,
            })

def add_path(source, stage):
    source = Path(source)
    if not source.exists():
        raise FileNotFoundError(source)
    destination = stage / source.relative_to(REPO_DIR)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        shutil.copytree(source, destination, dirs_exist_ok=True)
    else:
        shutil.copy2(source, destination)

def stage_archive(label, experiment_id, config_path, backend):
    stage = STAGE_DIR / label
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    work = REPO_DIR / "work" / experiment_id
    checkpoint = REPO_DIR / "checkpoint" / experiment_id
    common = [
        Path(config_path), work / "run_manifest.json", work / "selection_frozen.json",
        prediction_path(experiment_id, "val"), prediction_path(experiment_id, "test"),
        metric_path(experiment_id, "val"), metric_path(experiment_id, "test"),
    ]
    for optional in ("train.log", "training_history.json", "knowledge_augmentation.json"):
        if (work / optional).exists():
            common.append(work / optional)
    common.extend(sorted(work.glob("train.*.log")))
    if backend == "qlora":
        state = read_json(work / "trainer" / "trainer_state.json")
        checkpoints = [(int(path.name.rsplit("-", 1)[1]), path) for path in (work / "trainer").glob("checkpoint-*")]
        latest = max(checkpoints)[1]
        best = work / "trainer" / Path(state["best_model_checkpoint"]).name
        common.extend([checkpoint / "adapter", latest, work / "trainer" / "trainer_state.json"])
        if best != latest:
            common.append(best)
        curve = work / "training_curves.svg"
        if curve.exists():
            common.append(curve)
    else:
        common.append(checkpoint / "checkpoint_best.pt")
        if (work / "data-bin").is_dir():
            common.append(work / "data-bin")
    for source in common:
        add_path(source, stage)
    files = sorted(path for path in stage.rglob("*") if path.is_file())
    manifest = {
        "created_at": datetime.now(timezone.utc).isoformat(),
        "source_commit": RESOLVED_COMMIT,
        "experiment_id": experiment_id,
        "backend": backend,
        "files": {str(path.relative_to(stage)): sha256_file(path) for path in files},
    }
    (stage / f"stage_{label}_manifest.json").write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    archive_base = KAGGLE_WORKING / f"group10_{label}_results"
    archive = shutil.make_archive(str(archive_base), "zip", root_dir=stage)
    print("Created", archive)
    return Path(archive)

def partial_archive(label, experiment_id, config_path, backend):
    stage = STAGE_DIR / f"{label}_partial"
    if stage.exists():
        shutil.rmtree(stage)
    stage.mkdir(parents=True)
    candidates = [Path(config_path)]
    work = REPO_DIR / "work" / experiment_id
    checkpoint = REPO_DIR / "checkpoint" / experiment_id
    candidates.extend(sorted(work.glob("*.log")))
    for filename in ("run_manifest.json", "training_history.json"):
        if (work / filename).is_file():
            candidates.append(work / filename)
    candidates.extend(path for path in (work / "trainer").glob("checkpoint-*") if path.is_dir())
    if (work / "trainer/trainer_state.json").is_file():
        candidates.append(work / "trainer/trainer_state.json")
    if (work / "data-bin").is_dir():
        candidates.append(work / "data-bin")
    if backend == "qlora" and (checkpoint / "adapter").is_dir():
        candidates.append(checkpoint / "adapter")
    if backend != "qlora" and (checkpoint / "checkpoint_last.pt").is_file():
        candidates.append(checkpoint / "checkpoint_last.pt")
    for source in candidates:
        if source.exists():
            add_path(source, stage)
    archive = shutil.make_archive(str(KAGGLE_WORKING / f"group10_{label}_partial_results"), "zip", root_dir=stage)
    print("Created recoverable partial archive:", archive)


In [ ]:
def flattened_model_summary(label, config_path):
    cfg = yaml.safe_load(Path(config_path).read_text(encoding="utf-8"))
    training = cfg["training"]
    model = cfg.get("model", {})
    return {
        "experiment": label,
        "backend": cfg["backend"],
        "model": model.get("name", model.get("architecture")),
        "epochs/update limit": training.get("epochs", f"{training.get('max_epoch')} epochs / {training.get('max_update')} updates"),
        "learning_rate": training["learning_rate"],
        "effective_batch": training.get("per_device_batch_size", training.get("max_tokens_per_gpu")),
        "gradient_accumulation/update_freq": training.get("gradient_accumulation_steps", training.get("update_freq")),
        "precision": "bf16" if training.get("bf16") else ("fp16" if training.get("fp16") else "fp32"),
        "seed": cfg["seed"],
    }

model_config_df = pd.DataFrame([
    flattened_model_summary("E1 Fairseq", E1_CONFIG),
    flattened_model_summary("E2 Qwen3-8B QLoRA", E2_CONFIG),
    flattened_model_summary("Custom E3 retrieval", E3_CONFIG),
])
model_config_df.to_csv(TABLE_DIR / "model_configuration_summary.csv", index=False)
display(model_config_df)


### 4.1 E1 — Fairseq baseline

This establishes the corpus-only encoder-decoder baseline. A complete attached stage is restored and revalidated; otherwise the model is trained, selected on validation, evaluated, and archived.


In [ ]:
E1_ARCHIVE = run_fairseq_stage("e1", E1_ID, "e1_fairseq.yaml", RUN_E1, "fairseq")


In [ ]:
def parse_fairseq_validation(log_path):
    rows = []
    for line in Path(log_path).read_text(encoding="utf-8", errors="replace").splitlines():
        if " | valid | " not in line:
            continue
        try:
            payload = json.loads(line.split(" | valid | ", 1)[1])
        except json.JSONDecodeError:
            continue
        rows.append({key: float(payload[key]) for key in ("epoch", "valid_loss", "valid_bleu", "valid_num_updates") if key in payload})
    return pd.DataFrame(rows).drop_duplicates(subset=["epoch"], keep="last") if rows else pd.DataFrame()

def plot_fairseq_training(experiment_id, title, filename):
    history = parse_fairseq_validation(REPO_DIR / "work" / experiment_id / "train.log")
    if history.empty:
        print("No parseable validation history found for", experiment_id)
        return
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    axes[0].plot(history.epoch, history.valid_loss, marker="o", color="#457b9d")
    axes[0].set(title=f"{title}: validation loss", xlabel="Epoch", ylabel="Loss")
    axes[1].plot(history.epoch, history.valid_bleu, marker="o", color="#e76f51")
    axes[1].set(title=f"{title}: validation BLEU", xlabel="Epoch", ylabel="BLEU")
    save_figure(filename)

plot_fairseq_training(E1_ID, "E1", "e1_training_history.png")


### 4.2 Custom E3 — knowledge-augmented Fairseq

This controlled extension keeps E1's model and optimization settings while appending deterministic Classical Chinese candidate hints retrieved from `knowledge.json`. It is an original custom experiment, not the unavailable official E3.


In [ ]:
E3_ARCHIVE = run_fairseq_stage("e3_custom", E3_ID, "e3_custom_fairseq_knowledge.yaml", RUN_E3_CUSTOM, "fairseq_knowledge")


In [ ]:
plot_fairseq_training(E3_ID, "Custom E3", "e3_custom_training_history.png")


### 4.3 E2 — Qwen3-8B QLoRA

The preflight recovered and fingerprint-checked the attached run. This stage resumes only when the saved final epoch is below five, verifies the exact stopping epoch, regenerates validation/test outputs, and packages the adapter plus resumable state.


In [ ]:
if not completed_e2:
    try:
        run_command(sys.executable, "-m", "mt_pipeline", "train", "--config", E2_CONFIG)
    except subprocess.CalledProcessError:
        partial_archive("e2", E2_ID, E2_CONFIG, "qlora")
        raise
    final_state = read_json(REPO_DIR / "work" / E2_ID / "trainer" / "trainer_state.json")
    if abs(float(final_state["epoch"]) - E2_TARGET_EPOCHS) > 1e-6:
        raise RuntimeError(f"E2 stopped at epoch {final_state['epoch']}, expected {E2_TARGET_EPOCHS}")

# Always regenerate selection and predictions from the restored five-epoch adapter.
# This also repairs bundles whose earlier validation freeze predates the final resume.
run_command(sys.executable, "code/visualize_e2_training.py", REPO_DIR / "work" / E2_ID / "trainer" / "trainer_state.json", REPO_DIR / "work" / E2_ID / "training_curves.svg")
run_evaluation_sequence(E2_CONFIG, E2_ID, sys.executable)
validate_complete_stage(E2_CONFIG, E2_ID)
E2_ARCHIVE = stage_archive("e2", E2_ID, E2_CONFIG, "qlora")


In [ ]:
trainer_state = read_json(REPO_DIR / "work" / E2_ID / "trainer/trainer_state.json")
history = pd.DataFrame(trainer_state.get("log_history", []))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
train_history = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
eval_history = history.dropna(subset=["eval_loss"]) if "eval_loss" in history else pd.DataFrame()
if not train_history.empty:
    axes[0].plot(train_history.epoch, train_history.loss, alpha=.75, color="#457b9d")
axes[0].set(title="E2 training loss", xlabel="Epoch", ylabel="Loss")
if not eval_history.empty:
    axes[1].plot(eval_history.epoch, eval_history.eval_loss, marker="o", color="#e76f51")
axes[1].set(title="E2 validation loss", xlabel="Epoch", ylabel="Loss")
save_figure("e2_training_history.png")


## 5. Evaluation

All systems use `moses-char-v1`: NFC character-spaced Chinese, Sacremoses 0.2.0 (`lang=zh`, escaping and aggressive dash splitting disabled), then SacreBLEU 2.6.0 with `tokenize=none`. chrF++ keeps its original character-spaced inputs. Historical bundles are verified under their recorded protocol; fresh Moses metrics go to `metrics/moses/` without replacing their selection evidence. Training-time Fairseq BLEU remains the historical 13a selection metric. The tables separate validation from test. Seeded paired bootstrap resampling estimates uncertainty in test-set deltas. The 100-sentence manual sheet is pending until every system row has been adjudicated.


In [ ]:
MOSES_METRIC_DIR = REPO_DIR / "metrics" / "moses"
for experiment_id in (E1_ID, E2_ID, E3_ID):
    for split in ("val", "test"):
        run_command(sys.executable, "-m", "mt_pipeline", "evaluate", "--predictions", prediction_path(experiment_id, split), "--protocol", "moses-char-v1", "--output", MOSES_METRIC_DIR / f"{experiment_id}.{split}.json")

comparisons = (
    (E1_ID, E2_ID, "e1_vs_e2.test.json"),
    (E1_ID, E3_ID, "e1_vs_e3_custom.test.json"),
    (E3_ID, E2_ID, "e3_custom_vs_e2.test.json"),
)
for baseline, candidate, output in comparisons:
    run_command(sys.executable, "-m", "mt_pipeline", "compare", "--baseline", prediction_path(baseline, "test"), "--candidate", prediction_path(candidate, "test"), "--samples", "1000", "--seed", "42", "--protocol", "moses-char-v1", "--output", MOSES_METRIC_DIR / output)

annotation_path = REPO_DIR / "error_analysis/manual_sample.jsonl"
attached_annotations = []
for root in SEARCH_ROOTS:
    attached_annotations.extend(root.rglob("manual_sample.jsonl"))
attached_annotations = sorted({path.resolve() for path in attached_annotations})
if len(attached_annotations) > 1:
    raise RuntimeError(f"Multiple manual annotation sheets were attached: {attached_annotations}")
if attached_annotations:
    annotation_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(attached_annotations[0], annotation_path)
    print("Restored manual annotations from", attached_annotations[0])
else:
    run_command(sys.executable, "-m", "mt_pipeline", "prepare-error-analysis", "--predictions", prediction_path(E1_ID, "test"), prediction_path(E2_ID, "test"), prediction_path(E3_ID, "test"), "--sample-size", "100", "--seed", "42", "--output", annotation_path)

annotation_rows = [json.loads(line) for line in annotation_path.read_text(encoding="utf-8").splitlines() if line]
if annotation_rows and all(row.get("annotation_status") == "ADJUDICATED" for row in annotation_rows):
    run_command(sys.executable, "-m", "mt_pipeline", "summarize-error-analysis", "--annotations", annotation_path, "--output", REPO_DIR / "error_analysis/summary.json")
else:
    pending = sum(row.get("annotation_status") != "ADJUDICATED" for row in annotation_rows)
    print(f"Manual error analysis is pending: {pending}/{len(annotation_rows)} system rows need adjudication.")
run_command(sys.executable, "-m", "mt_pipeline", "project-status", "--configs", REPO_DIR / "configs/e1_fairseq.yaml", E2_CONFIG, REPO_DIR / "configs/e3_custom_fairseq_knowledge.yaml", "--output", REPO_DIR / "metrics/kaggle_project_status.json")


In [ ]:
experiment_labels = {
    E1_ID: "E1 Fairseq",
    E2_ID: "E2 Qwen3-8B QLoRA",
    E3_ID: "Custom E3 retrieval",
}
summary_rows = []
for experiment_id, label in experiment_labels.items():
    for split in ("val", "test"):
        metrics = read_json(MOSES_METRIC_DIR / f"{experiment_id}.{split}.json")["metrics"]
        summary_rows.append({
            "experiment_id": experiment_id, "experiment": label, "split": split,
            "BLEU": metrics["sacrebleu"]["score"], "chrF++": metrics["chrf_pp"]["score"],
            "protocol": metrics["sacrebleu"]["preprocessing"]["protocol"],
            "bleu_signature": metrics["sacrebleu"]["signature"],
            "bleu_preprocessing": json.dumps(metrics["sacrebleu"]["preprocessing"], sort_keys=True),
            "chrf_signature": metrics["chrf_pp"]["signature"],
        })
metrics_df = pd.DataFrame(summary_rows)
metrics_df.to_csv(TABLE_DIR / "evaluation_metrics.csv", index=False)
display(metrics_df.pivot(index="experiment", columns="split", values=["BLEU", "chrF++"]).round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for ax, metric in zip(axes, ("BLEU", "chrF++")):
    metrics_df.pivot(index="experiment", columns="split", values=metric).plot.bar(ax=ax, color=["#6d597a", "#355070"])
    ax.set(title=f"{metric} by split", xlabel="", ylabel=metric)
    ax.tick_params(axis="x", rotation=15)
save_figure("validation_and_test_metrics.png")


In [ ]:
comparison_files = {
    "E1 → E2": "e1_vs_e2.test.json",
    "E1 → custom E3": "e1_vs_e3_custom.test.json",
    "custom E3 → E2": "e3_custom_vs_e2.test.json",
}
bootstrap_rows = []
for comparison, filename in comparison_files.items():
    result = read_json(MOSES_METRIC_DIR / filename)
    for key, label in (("sacrebleu", "BLEU"), ("chrf_pp", "chrF++")):
        metric = result["metrics"][key]
        low, high = metric["delta_95_percent_interval"]
        bootstrap_rows.append({
            "comparison": comparison, "metric": label,
            "delta": metric["observed_delta"], "ci_low": low, "ci_high": high,
            "p_value": metric["two_sided_p_value"],
        })
bootstrap_df = pd.DataFrame(bootstrap_rows)
bootstrap_df.to_csv(TABLE_DIR / "paired_bootstrap_comparisons.csv", index=False)
display(bootstrap_df.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.4), sharey=True)
for ax, metric in zip(axes, ("BLEU", "chrF++")):
    frame = bootstrap_df[bootstrap_df.metric == metric].reset_index(drop=True)
    lower = frame.delta - frame.ci_low
    upper = frame.ci_high - frame.delta
    ax.errorbar(frame.delta, range(len(frame)), xerr=[lower, upper], fmt="o", capsize=4, color="#264653")
    ax.axvline(0, color="black", linewidth=1, linestyle="--")
    ax.set(title=f"Paired bootstrap: {metric}", xlabel="Candidate − baseline")
    ax.set_yticks(range(len(frame)), frame.comparison)
save_figure("paired_bootstrap_intervals.png")


In [ ]:
def read_prediction_rows(experiment_id):
    return [json.loads(line) for line in prediction_path(experiment_id, "test").read_text(encoding="utf-8").splitlines() if line]

predictions = {experiment_id: read_prediction_rows(experiment_id) for experiment_id in experiment_labels}
length_rows = []
for experiment_id, rows in predictions.items():
    for row in rows:
        reference_length = max(len("".join(row["reference"].split())), 1)
        prediction_length = len("".join(row["prediction"].split()))
        length_rows.append({
            "experiment": experiment_labels[experiment_id], "sample_id": row["sample_id"],
            "length_ratio": prediction_length / reference_length,
        })
length_ratio_df = pd.DataFrame(length_rows)
length_ratio_df.groupby("experiment").length_ratio.agg(["mean", "median", "std"]).to_csv(TABLE_DIR / "output_length_ratios.csv")

fig, ax = plt.subplots(figsize=(9, 4.2))
groups = [length_ratio_df.loc[length_ratio_df.experiment == label, "length_ratio"] for label in experiment_labels.values()]
ax.boxplot(groups, labels=list(experiment_labels.values()), showfliers=False)
ax.axhline(1.0, color="black", linestyle="--", linewidth=1)
ax.set(title="Test output/reference character-length ratios", ylabel="Length ratio")
ax.tick_params(axis="x", rotation=12)
save_figure("output_length_ratios.png")

by_system = {experiment_id: {row["sample_id"]: row for row in rows} for experiment_id, rows in predictions.items()}
shared_ids = sorted(set.intersection(*(set(rows) for rows in by_system.values())))
selected_ids = random.Random(EDA_SAMPLE_SEED).sample(shared_ids, min(QUALITATIVE_SAMPLE_SIZE, len(shared_ids)))
qualitative_rows = []
for sample_id in selected_ids:
    anchor = by_system[E1_ID][sample_id]
    qualitative_rows.append({
        "sample_id": sample_id, "Vietnamese source": anchor["source"], "reference": anchor["reference"],
        "E1 prediction": by_system[E1_ID][sample_id]["prediction"],
        "E2 prediction": by_system[E2_ID][sample_id]["prediction"],
        "custom E3 prediction": by_system[E3_ID][sample_id]["prediction"],
    })
qualitative_df = pd.DataFrame(qualitative_rows)
if SHOW_PRIVATE_EXAMPLES:
    display(qualitative_df)
else:
    print("Qualitative predictions hidden because SHOW_PRIVATE_EXAMPLES=False.")


### 5.1 Automatic rule-based error diagnostics

The root-level `Error_Analysis.py` is run on the same seeded 100 test IDs for all three systems. It applies synonym normalization, technical-failure checks for code-switching and degenerate repetition, number/time checks, a fixed entity list, character-overlap thresholds, and rule-based omission/addition detection. The two technical checks short-circuit to `OTHER`, so a collapsed output is not also counted as an omission or an unsupported addition. These labels are useful for exploratory diagnosis, but they are **heuristic proxy labels** derived from a single reference—not human judgments and not a substitute for the adjudicated annotation sheet below.


In [ ]:
AUTOMATIC_ERROR_INPUT = REPO_DIR / "Error_Analysis_PreLabeled.xlsx"
AUTOMATIC_ERROR_OUTPUT = REPO_DIR / "Error_Analysis_Labeled.xlsx"
ERROR_LABELS = [
    "CORRECT", "MISTRANSLATION", "OMISSION", "UNSUPPORTED_ADDITION",
    "ENTITY_ERROR", "NUMBER_OR_TIME_ERROR", "OTHER",
]
sheet_names = {E1_ID: "E1", E2_ID: "E2", E3_ID: "E3_Custom"}

# Build exactly 100 aligned rows per system from the shared annotation sample.
with pd.ExcelWriter(AUTOMATIC_ERROR_INPUT, engine="openpyxl") as writer:
    for experiment_id, sheet_name in sheet_names.items():
        system_rows = [row for row in annotation_rows if row["experiment_id"] == experiment_id]
        if len(system_rows) != 100:
            raise RuntimeError(f"{experiment_id} has {len(system_rows)} shared error-analysis rows, expected 100")
        frame = pd.DataFrame([
            {
                "Sample ID": row["sample_id"],
                "Source": row["source"],
                "Reference": row["reference"],
                "Prediction": row["prediction"],
            }
            for row in system_rows
        ])
        frame.to_excel(writer, sheet_name=sheet_name, index=False)

run_command(sys.executable, ERROR_ANALYSIS_SCRIPT, cwd=REPO_DIR)
if not AUTOMATIC_ERROR_OUTPUT.is_file():
    raise RuntimeError("Error_Analysis.py did not create Error_Analysis_Labeled.xlsx")

automatic_rows = []
for experiment_id, sheet_name in sheet_names.items():
    frame = pd.read_excel(AUTOMATIC_ERROR_OUTPUT, sheet_name=sheet_name)
    frame = frame[frame["Sample ID"].notna()].copy()
    frame["experiment_id"] = experiment_id
    frame["experiment"] = experiment_labels[experiment_id]
    automatic_rows.append(frame)
automatic_error_df = pd.concat(automatic_rows, ignore_index=True)
for label in ERROR_LABELS:
    automatic_error_df[label] = pd.to_numeric(automatic_error_df[label], errors="coerce").fillna(0).astype(int)

automatic_summary_df = (
    automatic_error_df.groupby(["experiment_id", "experiment"])[ERROR_LABELS]
    .mean().mul(100).reset_index()
)
automatic_summary_df.to_csv(TABLE_DIR / "automatic_error_analysis_summary.csv", index=False)
automatic_error_df.to_csv(TABLE_DIR / "automatic_error_analysis_details.csv", index=False)
display(automatic_summary_df.round(2))

plot_frame = automatic_summary_df.set_index("experiment")[ERROR_LABELS].T
ax = plot_frame.plot.bar(figsize=(14, 5), width=0.8)
ax.set(
    title="Automatic rule-based error-label prevalence on shared 100-sentence sample",
    xlabel="Heuristic label",
    ylabel="Sentences labelled (%)",
)
ax.tick_params(axis="x", rotation=25)
ax.legend(title="System")
save_figure("automatic_error_analysis.png")
print("Automatic labels may overlap, so percentages are not expected to sum to 100%.")


In [ ]:
test_metrics = metrics_df[metrics_df.split == "test"].sort_values("BLEU", ascending=False)
best = test_metrics.iloc[0]
significant = bootstrap_df[(bootstrap_df.ci_low > 0) | (bootstrap_df.ci_high < 0)]
print(f"Highest observed test BLEU: {best['experiment']} ({best['BLEU']:.3f}); this is dataset-specific evidence, not a universal ranking.")
print(f"{len(significant)}/{len(bootstrap_df)} metric comparisons have 95% bootstrap intervals that exclude zero.")

summary_path = REPO_DIR / "error_analysis/summary.json"
if summary_path.exists():
    error_summary = read_json(summary_path)["experiments"]
    error_rows = []
    for experiment, payload in error_summary.items():
        for label, percentage in payload["label_percentages"].items():
            error_rows.append({"experiment": experiment_labels.get(experiment, experiment), "label": label, "percentage": percentage})
    error_df = pd.DataFrame(error_rows)
    error_df.to_csv(TABLE_DIR / "manual_error_analysis.csv", index=False)
    pivot = error_df.pivot(index="label", columns="experiment", values="percentage").fillna(0)
    ax = pivot.plot.bar(figsize=(12, 4.8))
    ax.set(title="Adjudicated manual error labels", xlabel="", ylabel="Percent of 100 sampled sentences")
    ax.tick_params(axis="x", rotation=25)
    save_figure("manual_error_analysis.png")
else:
    print("Manual error chart not generated: complete and adjudicate error_analysis/manual_sample.jsonl, then rerun this section.")


### Evaluation limitations

- The corpus is small, domain-specific, and has silver training alignment; findings may not transfer beyond DVSKTT-style historical text.
- Six training targets contain replacement characters and required duplicate pairs remain for experimental comparability.
- BLEU and chrF++ reward surface overlap and cannot fully judge historical fidelity, ambiguity, or acceptable variants.
- Three systems on one fixed 510-sentence test split provide limited evidence; bootstrap intervals quantify sampling variation, not every source of uncertainty.
- Custom E3 tests one deterministic knowledge-injection design and must not be conflated with the unavailable official E3.
- Manual error conclusions are withheld until all sampled outputs are independently annotated and adjudicated.


## 6. Deployment and private handoff

The final step writes project status, combines every stage, adds the audit, comparisons, manual-error sheet or summary, CRISP-DM figures/tables, a source-commit manifest, and SHA-256 checksums, then creates private zip files in `/kaggle/working`. Download the final archive plus stage archives before the session ends. A partial archive is produced automatically if training fails, allowing a later private Kaggle session to resume.


In [ ]:
final_stage = STAGE_DIR / "all"
if final_stage.exists():
    shutil.rmtree(final_stage)
final_stage.mkdir(parents=True)

# Rebuild a stage directory if an earlier packaging cell did not materialize it.
stage_specs = (
    ("e1", E1_ID, E1_CONFIG, "fairseq"),
    ("e2", E2_ID, E2_CONFIG, "qlora"),
    ("e3_custom", E3_ID, E3_CONFIG, "fairseq_knowledge"),
)
for label, experiment_id, config_path, backend in stage_specs:
    stage = STAGE_DIR / label
    if not stage.is_dir():
        print(f"Stage directory is missing; rebuilding: {stage}")
        rebuilt_archive = stage_archive(label, experiment_id, config_path, backend)
        if label == "e1":
            E1_ARCHIVE = rebuilt_archive
        elif label == "e2":
            E2_ARCHIVE = rebuilt_archive
        else:
            E3_ARCHIVE = rebuilt_archive
    shutil.copytree(stage, final_stage, dirs_exist_ok=True)
for path in (
    REPO_DIR / "metrics/data_audit.json",
    MOSES_METRIC_DIR,
    REPO_DIR / "metrics/kaggle_project_status.json",
    REPO_DIR / "error_analysis/manual_sample.jsonl",
    REPO_DIR / "Error_Analysis.py",
    REPO_DIR / "Error_Analysis_PreLabeled.xlsx",
    REPO_DIR / "Error_Analysis_Labeled.xlsx",
    REPO_DIR / "analysis/figures",
    REPO_DIR / "analysis/tables",
):
    add_path(path, final_stage)
if (REPO_DIR / "error_analysis/summary.json").exists():
    add_path(REPO_DIR / "error_analysis/summary.json", final_stage)

run_manifest = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "source_url": REPO_URL,
    "source_ref": REPO_REF,
    "source_commit": RESOLVED_COMMIT,
    "experiments": [E1_ID, E2_ID, E3_ID],
    "e2_target_epochs": E2_TARGET_EPOCHS,
    "methodology": "CRISP-DM",
    "evaluation_protocol": "moses-char-v1",
    "evaluation_metrics": "metrics/moses",
    "automatic_error_analysis": "Error_Analysis.py heuristic labels on shared seeded 100-sentence sample",
    "figures": sorted(path.name for path in FIGURE_DIR.glob("*.png")),
    "tables": sorted(path.name for path in TABLE_DIR.glob("*.csv")),
    "official_e3_status": "blocked; custom E3 executed under a distinct ID",
    "privacy": "Restricted corpus, predictions, derived binaries, and models must remain private.",
}
(final_stage / "kaggle_run_manifest.json").write_text(json.dumps(run_manifest, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
(final_stage / "PRIVATE_README.txt").write_text(
    "Private Group 10 E1-E3 Kaggle artifacts. Do not publish this archive: it contains model outputs, predictions, and derived course data.\n",
    encoding="utf-8",
)
final_files = sorted(path for path in final_stage.rglob("*") if path.is_file())
(final_stage / "SHA256SUMS.json").write_text(json.dumps({str(path.relative_to(final_stage)): sha256_file(path) for path in final_files}, indent=2, sort_keys=True) + "\n", encoding="utf-8")
FINAL_ARCHIVE = Path(shutil.make_archive(str(KAGGLE_WORKING / "group10_e1_e3_kaggle_results"), "zip", root_dir=final_stage))

print("Final private archive:", FINAL_ARCHIVE)
for archive in (E1_ARCHIVE, E2_ARCHIVE, E3_ARCHIVE, FINAL_ARCHIVE):
    print(f"  {archive.name}: {archive.stat().st_size / 1024**3:.2f} GiB  sha256={sha256_file(archive)}")
print("Keep every archive and this notebook private.")
